# Oakland Coliseum Game & Weather Dataset (2015-2024)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Oakland Coliseum. It combines:
1. **Game statistics** from `master_data.csv` (pre-aggregated Statcast data)
2. **Weather data** from Open-Meteo hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

Note: The Athletics moved from Oakland after the 2024 season, so this dataset covers 2015-2024.

## Section 0: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# === STADIUM CONFIGURATION ===
STADIUM_NAME = 'Oakland Coliseum'
HOME_TEAM = 'ATH'
SEASONS = range(2015, 2025)  # 2015 through 2024
TIMEZONE = 'America/Los_Angeles'

# Coordinates
STADIUM_LAT = 37.7514
STADIUM_LON = -122.2009

# Outfield directions (degrees from north)
CF_DIR = 45.0    # Center field: NE
LCF_DIR = 25.0   # Left-center field: NNE
RCF_DIR = 65.0   # Right-center field: ENE

# Output file
OUTPUT_FILE = 'athletics_data_2015.csv'

print(f"Configuration: {STADIUM_NAME}")
print(f"Home team: {HOME_TEAM}")
print(f"Seasons: {list(SEASONS)}")
print(f"Timezone: {TIMEZONE}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Configuration: Oakland Coliseum
Home team: ATH
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Timezone: America/Los_Angeles


## Section 1: Load & Filter Master Data

Read game-level statistics from `master_data.csv` and filter to Oakland Coliseum home games.

In [2]:
# Load master dataset
master = pd.read_csv(os.path.join('..', 'Final Datasets', 'master_data.csv'))
print(f"Master dataset: {len(master)} total games")

# Filter to this stadium's home games and season range
games = master[
    (master['home_team'] == HOME_TEAM) &
    (master['season'].isin(SEASONS))
].copy()

# Convert game_start_utc to local timezone
games['game_start'] = (
    pd.to_datetime(games['game_start_utc'], utc=True)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)  # Remove timezone info for clean processing
)
games['start_hour'] = games['game_start'].dt.hour
games['game_date'] = pd.to_datetime(games['game_date'])

# Drop master-only columns not needed in final output
games = games.drop(columns=['home_team', 'game_start_utc'])

games = games.sort_values('game_date').reset_index(drop=True)

print(f"\n{STADIUM_NAME} games: {len(games)}")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Master dataset: 25155 total games

Oakland Coliseum games: 757
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Games per season:
season
2015    81
2016    80
2017    80
2018    81
2019    81
2020    30
2021    81
2022    81
2023    81
2024    81
Name: game_pk, dtype: int64

Sample:
   game_pk  game_date  season away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  barrel_rate  \
0   413662 2015-04-06    2015       TEX                 8                 0           8              2          14      5    12            287               91.2          3     51       0.0588   
1   413670 2015-04-07    2015       TEX                 1                 3           4              0          13      3    13            260               85.3          1     51       0.0196   
2   413684 2015-04-08    2015       TEX                10                 0          10              0    

## Section 2: Pull Weather Data (Open-Meteo)

Use Open-Meteo to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [3]:
# Using Open-Meteo Historical Weather API (free, no key required)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': STADIUM_LAT,
    'longitude': STADIUM_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': TIMEZONE,
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}\u00b0C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}\u00b0")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 29.4°C
Sample wind: 8.8 km/h from 279°


In [4]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': STADIUM_LAT,
        'longitude': STADIUM_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': TIMEZONE,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive local time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2015...
  2015: 6600 hourly records
Pulling weather for 2016...
  2016: 6600 hourly records
Pulling weather for 2017...
  2017: 6600 hourly records
Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records

Weather records: 757
Missing temp data: 23
      temp_c       rhum         pres  prcp       wspd        wdir  game_pk
0  11.933333  66.333333  1010.566667   0.0  16.300000  179.997145   413662
1  10.166667  74.666667  1015.066667   0.0  10.266667  255.665006   413670
2  11.533333  74.000000  1018.566667   0.0   8.466667  297.333465   413684


## Section 3: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Oakland Coliseum outfield directions (degrees from north):
- Center field: ~45° (NE)
- Left-center field: ~25° (NNE)
- Right-center field: ~65° (ENE)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [5]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

Wind projection summary (positive = blowing out, negative = blowing in):
          wind_cf    wind_lcf    wind_rcf
count  734.000000  734.000000  734.000000
mean    10.130095    7.683475   11.354876
std      5.299856    4.997723    5.679924
min    -13.443040  -18.660480  -11.873338
25%      7.051253    4.728552    7.437715
50%     10.464064    8.078166   11.711150
75%     13.515478   10.775042   15.084199
max     23.919341   20.463467   28.064781


## Section 4: Final Assembly

Convert units, order columns, and round to sensible precision.

In [6]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

athletics_data = games_full[final_columns].copy()
athletics_data = athletics_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    athletics_data[col] = athletics_data[col].round(decimals)

print(f"Final dataset: {athletics_data.shape[0]} rows x {athletics_data.shape[1]} columns")

Final dataset: 757 rows x 31 columns


## Section 5: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [7]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = athletics_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 100)  # Normal: ~81 home games (wider range for doubleheaders)
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(athletics_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = athletics_data[col].isna().sum()
    pct = 100 * n_null / len(athletics_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', athletics_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', athletics_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', athletics_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', athletics_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', athletics_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', athletics_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', athletics_data['temp_f'].mean(), '~60-70 F'),
    ('Min game temp', athletics_data['temp_f'].min(), '>40 F'),
    ('Max game temp', athletics_data['temp_f'].max(), '<105 F'),
    ('Avg wind speed (km/h)', athletics_data['wspd'].mean(), '~10-20 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(athletics_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2015: 81 games [OK] (expected 75-100)
  2016: 80 games [OK] (expected 75-100)
  2017: 80 games [OK] (expected 75-100)
  2018: 81 games [OK] (expected 75-100)
  2019: 81 games [OK] (expected 75-100)
  2020: 30 games [OK] (expected 25-35)
  2021: 81 games [OK] (expected 75-100)
  2022: 81 games [OK] (expected 75-100)
  2023: 81 games [OK] (expected 75-100)
  2024: 81 games [OK] (expected 75-100)
  TOTAL: 757 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 3 nulls (0.4%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 23 nulls (3.0%) [OK]
  wspd: 23 nulls (3.0%) [OK]
  wdir: 23 nulls (3.0%) [OK]
  wind_cf: 23 nulls (3.0%) [OK]
  game_start: 23 nulls (3.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 8.57 (expected ~8-10)
  Avg HR/game: 2.11 (expected ~2-3)
  Avg K/ga

In [8]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
athletics_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,413662,2015-04-06,2015,TEX,2015-04-06 19:05:00,19.0,8,0,8,2,14,5,12,287,91.2,3,51,0.0588,0.1667,53.5,11.9,66.3,1010.6,0.0,16.3,10.1,180.0,S,11.53,14.77,6.89
1,413670,2015-04-07,2015,TEX,2015-04-07 19:05:00,19.0,1,3,4,0,13,3,13,260,85.3,1,51,0.0196,0.0000,50.3,10.2,74.7,1015.1,0.0,10.3,6.4,255.7,W,8.83,6.51,10.09
2,413684,2015-04-08,2015,TEX,2015-04-08 19:05:00,19.0,10,0,10,0,17,5,18,286,85.9,1,53,0.0189,0.0000,52.8,11.5,74.0,1018.6,0.0,8.5,5.3,297.3,NW,2.57,-0.34,5.17
3,413694,2015-04-09,2015,TEX,2015-04-09 12:35:00,12.0,1,10,11,4,11,5,18,274,84.6,3,59,0.0508,0.2222,64.3,17.9,46.3,1014.8,0.0,1.7,1.1,324.1,NW,-0.27,-0.84,0.33
4,413705,2015-04-10,2015,SEA,2015-04-10 19:05:00,19.0,12,0,12,1,13,6,19,288,85.9,2,56,0.0357,0.0526,54.9,12.7,71.3,1016.1,0.0,5.5,3.4,249.7,W,4.97,3.89,5.45
5,413720,2015-04-11,2015,SEA,2015-04-11 13:05:00,13.0,4,5,9,2,13,6,22,336,87.5,4,74,0.0541,0.0909,69.1,20.6,46.3,1016.4,0.0,17.5,10.9,278.7,W,10.37,4.92,14.56
6,413735,2015-04-12,2015,SEA,2015-04-12 13:05:00,13.0,7,8,15,2,11,7,21,303,87.0,1,68,0.0147,0.0952,73.9,23.3,37.7,1014.7,0.0,9.0,5.6,286.7,W,4.29,1.31,6.75
7,413889,2015-04-24,2015,HOU,2015-04-24 19:05:00,19.0,4,5,9,1,15,10,19,348,90.0,2,66,0.0303,0.0526,53.9,12.2,65.3,1015.1,0.0,13.7,8.5,212.6,SW,13.42,13.61,11.60
8,413904,2015-04-25,2015,HOU,2015-04-25 13:05:00,13.0,3,9,12,2,13,8,17,294,86.3,5,58,0.0862,0.1176,61.4,16.3,56.3,1014.2,0.0,19.1,11.9,273.0,W,12.78,7.15,16.86
9,413919,2015-04-26,2015,HOU,2015-04-26 13:05:00,13.0,6,7,13,1,15,7,17,306,88.7,2,57,0.0351,0.0588,70.7,21.5,40.0,1018.6,0.0,16.2,10.1,281.7,W,8.90,3.74,12.99


## Section 6: Save to CSV

In [9]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, OUTPUT_FILE)
athletics_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(athletics_data)}, Columns: {len(athletics_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == athletics_data.shape, f"Shape mismatch: {verify.shape} vs {athletics_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/athletics_data_2015.csv
File size: 113.2 KB
Rows: 757, Columns: 31

Save & reload verification: PASSED
